# 06 – Paired bootstrap, 10,000 resamples
Computes the 95% confidence intervals and the paired two-sided p-values reported in the paper,
with B = 10,000 image-level resamples (seed 12345), using the shared `rdd_fair.fair_eval` evaluator
and the detections saved by notebook 04 (`results/<model>/seed0/predictions/`).

The outputs below are from the run used for the paper. Results file: `results/paired_bootstrap_10k_results.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1) Find the rdd_fair package inside My Drive/RDD_fair automatically
import sys, glob, os
BASE = '/content/drive/MyDrive/RDD_fair'
hits = glob.glob(f'{BASE}/**/rdd_fair/fair_eval.py', recursive=True)
print(hits)
assert hits, 'fair_eval.py not found under RDD_fair - check the folder name'
sys.path.insert(0, os.path.dirname(os.path.dirname(hits[0])))
from rdd_fair.fair_eval import Evaluator, IOU_THRS_COCO
print('rdd_fair imported from', os.path.dirname(hits[0]))

['/content/drive/MyDrive/RDD_fair/code/rdd_fair/fair_eval.py']
rdd_fair imported from /content/drive/MyDrive/RDD_fair/code/rdd_fair


In [ ]:
# 2) Software versions of this run
!python --version
!nvidia-smi | head -4
from importlib.metadata import version
for pkg in ['torch', 'torchvision', 'pycocotools', 'ultralytics']:
    try: print(pkg, version(pkg))
    except Exception: print(pkg, 'not installed')
import torch; print('CUDA', torch.version.cuda)

Python 3.13.15
Mon Oct 12 22:52:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
torch 2.11.0+cu130
torchvision 0.26.0+cu130
pycocotools 2.0.11
ultralytics 8.4.176
CUDA 13.0


In [ ]:
# 3) Paths
R = '/content/drive/MyDrive/RDD_fair/results'
GT = {
    'RDD2022_test': f'{R}/00_dataset/rdd_test_coco.json',
    'Albania':      f'{R}/00_dataset/albania_coco.json',
}
MODEL_DIR = {'YOLOv8m': 'yolov8m', 'SSD300-VGG16': 'ssd300_vgg16', 'Faster R-CNN': 'fasterrcnn_r50_fpn_v2'}
DETS = {
    'RDD2022_test': {m: f'{R}/{d}/seed0/predictions/test.json'    for m, d in MODEL_DIR.items()},
    'Albania':      {m: f'{R}/{d}/seed0/predictions/albania.json' for m, d in MODEL_DIR.items()},
}
# Validation-tuned thresholds (Section 3.4)
THR = {'YOLOv8m': 0.20, 'SSD300-VGG16': 0.26, 'Faster R-CNN': 0.77}

# check: every file exists, and the saved thresholds match the ones above
import os, json
for ds in DETS:
    assert os.path.exists(GT[ds]), GT[ds]
    for p in DETS[ds].values(): assert os.path.exists(p), p
for m, d in MODEL_DIR.items():
    print(m, THR[m], '| saved threshold:', json.load(open(f'{R}/{d}/seed0/metrics/operating_threshold.json'))['threshold'])
print('all paths OK')

YOLOv8m 0.2 | saved threshold: 0.2
SSD300-VGG16 0.26 | saved threshold: 0.26
Faster R-CNN 0.77 | saved threshold: 0.77
all paths OK


In [ ]:
import numpy as np
from rdd_fair.fair_eval import IOU_THRS_COCO

B = 10_000
SEED = 12345  # same seed as in the paper


def _metrics(ev, tables, thr, w):
    """mAP@0.5, mAP@0.5:0.95, macro-F1 and per-class AP@0.5 for one bootstrap weight vector."""
    ap = ev.ap_from_tables(tables, w, iou_subset=IOU_THRS_COCO)          # {cid: AP over 10 IoUs}
    arr = np.array(list(ap.values()))                                    # K x 10
    out = {"mAP50": np.nanmean(arr[:, 0]),
           "mAP50_95": np.nanmean(arr),
           "macro_F1": ev.pr_f1(tables, thr, img_weights=w)["macro"]["F1"]}
    for k, cid in enumerate(ev.cat_ids):
        out[f"{ev.cat_names[cid]}_AP50"] = arr[k, 0]
    return out


def paired_test(ev, dets_a, thr_a, dets_b, thr_b, B=B, seed=SEED):
    """Two-sided paired bootstrap p-values for model A minus model B.
    Both models are scored on the SAME resampled images in every iteration."""
    ta, tb = ev.match_tables(dets_a), ev.match_tables(dets_b)
    n = len(ev.img_ids)
    point_a = _metrics(ev, ta, thr_a, np.ones(n))
    point_b = _metrics(ev, tb, thr_b, np.ones(n))

    rng = np.random.default_rng(seed)
    diffs = {k: [] for k in point_a}
    samples_a = {k: [] for k in point_a}
    samples_b = {k: [] for k in point_a}
    for i in range(B):
        w = np.bincount(rng.integers(0, n, n), minlength=n).astype(float)
        ma, mb = _metrics(ev, ta, thr_a, w), _metrics(ev, tb, thr_b, w)
        for k in point_a:
            diffs[k].append(ma[k] - mb[k])
            samples_a[k].append(ma[k])
            samples_b[k].append(mb[k])
        if (i + 1) % 1000 == 0:
            print(f"{i + 1}/{B}")

    res = {}
    for k, v in diffs.items():
        v = np.asarray(v, float)
        v = v[~np.isnan(v)]
        # two-sided p: how often the bootstrap difference falls on the other side of zero
        p = min(1.0, 2 * min(np.mean(v <= 0), np.mean(v >= 0)))
        ci = lambda s: np.nanpercentile(np.asarray(s, float), [2.5, 97.5])
        res[k] = {"A": point_a[k], "B": point_b[k], "diff": point_a[k] - point_b[k],
                  "p": p, "diff_CI95": np.percentile(v, [2.5, 97.5]).tolist(),
                  "A_CI95": ci(samples_a[k]).tolist(), "B_CI95": ci(samples_b[k]).tolist()}
    return res


def show(res):
    for k, r in res.items():
        print(f"{k:12s} A={r['A']:.2f} B={r['B']:.2f} diff={r['diff']:+.2f} "
              f"p={r['p']:.4f}  A 95% CI {r['A_CI95'][0]:.1f}-{r['A_CI95'][1]:.1f}  "
              f"B 95% CI {r['B_CI95'][0]:.1f}-{r['B_CI95'][1]:.1f}")

## Quick timing test
Runs 1,000 resamples for one pair first so you can see how long 10,000 will take (roughly 10x).

In [ ]:
import json, time
ev_alb = Evaluator(GT['Albania'])
d = {m: json.load(open(p)) for m, p in DETS['Albania'].items()}
t0 = time.time()
_ = paired_test(ev_alb, d['Faster R-CNN'], THR['Faster R-CNN'], d['YOLOv8m'], THR['YOLOv8m'], B=1000)
print(f'1,000 resamples took {time.time()-t0:.0f} s')

1000/1000
1,000 resamples took 10 s


## Full run, B = 10,000
Values are in percent (e.g. 57.8 = 0.578). `diff` and `p` are model A minus model B. Results are also saved to Drive as JSON.

In [ ]:
PAIRS = [('Faster R-CNN', 'YOLOv8m'), ('Faster R-CNN', 'SSD300-VGG16'), ('YOLOv8m', 'SSD300-VGG16')]
OUT = '/content/drive/MyDrive/RDD_fair/results/paired_bootstrap_10k_results.json'

all_res = {}
for ds in ['Albania', 'RDD2022_test']:
    ev = Evaluator(GT[ds])
    d = {m: json.load(open(p)) for m, p in DETS[ds].items()}
    for a, b in PAIRS:
        print(f'\n=== {ds}: {a} vs {b} ===')
        r = paired_test(ev, d[a], THR[a], d[b], THR[b])
        show(r)
        all_res[f'{ds} | {a} vs {b}'] = r
        json.dump(all_res, open(OUT, 'w'), indent=1, default=float)  # save after each pair
print('saved to', OUT)


=== Albania: Faster R-CNN vs YOLOv8m ===
1000/10000
2000/10000
3000/10000
4000/10000
5000/10000
6000/10000
7000/10000
8000/10000
9000/10000
10000/10000
mAP50        A=29.24 B=24.69 diff=+4.55 p=0.0120  A 95% CI 24.5-35.0  B 95% CI 20.6-30.2
mAP50_95     A=15.31 B=12.10 diff=+3.20 p=0.0132  A 95% CI 12.2-19.3  B 95% CI 9.6-15.7
macro_F1     A=33.59 B=30.16 diff=+3.43 p=0.0602  A 95% CI 28.9-38.3  B 95% CI 25.5-34.7
D00_AP50     A=17.82 B=13.30 diff=+4.52 p=0.0170  A 95% CI 13.5-23.5  B 95% CI 9.7-18.9
D10_AP50     A=20.97 B=23.14 diff=-2.17 p=0.1898  A 95% CI 15.8-27.1  B 95% CI 18.4-28.9
D20_AP50     A=27.97 B=28.73 diff=-0.77 p=0.7956  A 95% CI 19.4-37.9  B 95% CI 20.4-38.1
D40_AP50     A=50.20 B=33.59 diff=+16.62 p=0.0090  A 95% CI 36.0-65.6  B 95% CI 21.1-48.9

=== Albania: Faster R-CNN vs SSD300-VGG16 ===
1000/10000
2000/10000
3000/10000
4000/10000
5000/10000
6000/10000
7000/10000
8000/10000
9000/10000
10000/10000
mAP50        A=29.24 B=18.85 diff=+10.39 p=0.0000  A 95% CI 24.5-35

## Values used in the paper
The 95% confidence intervals in the captions of Tables 4 and 7 (`A_CI95` / `B_CI95` for mAP@0.5 and macro-F1)
and all paired p-values in Sections 4.2, 4.4, 4.6 and the Conclusions come from the run above
(B = 10,000, seed 12345). Values above are in percent.